# CropIQ Phase 4: Actionable Agricultural Recommendation Engine

**Subtitle:** AI-Powered Crop Yield Intelligence  
**Tagline:** Predict. Understand. Optimize.  
**Objective:** Transform Phase 2 ML predictions and Phase 3 explainability & risk signals into deterministic, prioritized, and farmer-friendly agricultural recommendations.

```text
Farm Input
    ↓
Phase 2 — Yield Prediction
    ↓
Phase 3 — Explainability + Risk + Uncertainty
    ↓
Phase 4 — Recommendation Engine
    ↓
Prioritized Actions & Monitoring Suggestions
    ↓
Phase 5 — What-If Simulation
```

## 1. Imports & Environment Setup
Import required data, intelligence, and recommendation engine modules.

In [ ]:
import sys
import json
from pathlib import Path
import pandas as pd
import numpy as np

# Ensure project root is in sys.path
for p in [Path.cwd(), Path.cwd().parent]:
    if (p / 'src').exists() and str(p) not in sys.path:
        sys.path.insert(0, str(p))

# CropIQ Core Modules
from src.intelligence import analyze_crop_prediction
from src.recommendations import (
    generate_recommendations,
    load_agricultural_rules,
    load_crop_profiles,
    load_recommendation_metadata,
    validate_rule_catalog,
    validate_single_recommendation,
    evaluate_candidate_rules,
    prioritize_and_rank_recommendations,
    format_recommendation_farmer_mode,
    format_recommendation_technical_mode,
    RuleTracer,
)

print('CropIQ Phase 4 modules imported successfully.')

CropIQ Phase 4 modules imported successfully.


## 2. Load Phase 3 Intelligence Pipeline
Load a realistic test observation from the processed dataset and generate Phase 3 intelligence.

In [ ]:
data_path = Path('data/processed/crop_yield_model_data.csv')
if not data_path.exists():
    data_path = Path('../data/processed/crop_yield_model_data.csv')

df = pd.read_csv(data_path)
sample_input = df.iloc[0].to_dict()
sample_input.pop('yield', None)

p3_payload = analyze_crop_prediction(sample_input)
print('Predicted Yield:', p3_payload['prediction'])
print('Risk Posture:', p3_payload['risk']['level'], f"(Score: {p3_payload['risk']['score']}/100)")
print('Ensemble Uncertainty:', p3_payload['uncertainty']['classification'])

Predicted Yield: {'yield': 36.4261, 'unit': 'unconfirmed'}
Risk Posture: MODERATE (Score: 59/100)
Ensemble Uncertainty: HIGH


## 3. Load Agricultural Knowledge Base
Load the external rules catalog, supported crop profiles, and recommendation metadata.

In [ ]:
rules = load_agricultural_rules()
crop_profiles = load_crop_profiles()
metadata = load_recommendation_metadata()

print(f'Loaded {len(rules)} agricultural rules across {len(metadata["categories"])} categories.')
print(f'Supported Crop Profiles: {list(crop_profiles.keys())}')

Loaded 19 agricultural rules across 9 categories.
Supported Crop Profiles: ['Rice', 'Wheat', 'Maize', 'Cotton', 'Pulses', 'Sugarcane', 'Soybean', 'Groundnut', 'Mustard', 'Millets']


## 4. Validate Rules & Safety Schema
Perform static safety audit: ensure unique IDs, valid categories, citation tags, and zero prohibited causal language.

In [ ]:
is_valid, issues = validate_rule_catalog(rules)
print('Knowledge Base Catalog Valid:', is_valid)
if not is_valid:
    for iss in issues:
        print(' -', iss)
else:
    print('Zero safety violations or unbacked dosage prescriptions detected.')

Knowledge Base Catalog Valid: True
Zero safety violations or unbacked dosage prescriptions detected.


## 5. Evaluate Triggers Against Farm Observation
Apply composite boolean logic, feature availability filters, and crop specificity.

In [ ]:
tracer = RuleTracer()
candidate_recs = evaluate_candidate_rules(
    intelligence_payload=p3_payload,
    input_data=sample_input,
    rules=rules,
    tracer=tracer,
)
print(f'Matched {len(candidate_recs)} candidate rules out of {len(rules)} total rules:')
for c in candidate_recs:
    print(f"  [{c['category']}] {c['id']}: {c['title']} (Base Priority: {c['priority']})")

Matched 5 candidate rules out of 19 total rules:
  [VEGETATION] veg_001: Monitor canopy vigor and conduct field scouting (Base Priority: MEDIUM)
  [SOIL] soil_001: Consider standard soil fertility testing (Base Priority: LOW)
  [RISK] risk_002: Moderate yield risk alert: Review key downward factors (Base Priority: MEDIUM)
  [MONITORING] uncertainty_001: High model uncertainty: Verify field conditions before major decisions (Base Priority: HIGH)
  [DATA_QUALITY] dq_002: Verify flagged data values to improve estimate reliability (Base Priority: MEDIUM)


## 6. Candidate Recommendations Inspection
Inspect attached empirical evidence, citations, and what-if simulation compatibility flags.

In [ ]:
if candidate_recs:
    c0 = candidate_recs[0]
    print('Rule ID:', c0['id'])
    print('Reason:', c0['reason'])
    print('Action:', c0['action'])
    print('What-If Supported:', c0['what_if_supported'])
    print('Attached Evidence Items:', len(c0['evidence']))

Rule ID: veg_001
Reason: Vegetation index (NDVI) is lower than average and pulled down the model prediction relative to baseline.
Action: Conduct targeted field walkthroughs to check canopy greenness, stand uniformity, and potential biotic or abiotic stressors.
What-If Supported: False
Attached Evidence Items: 2


## 7 & 8. Deduplication and Agronomic Conflict Resolution
Consolidate overlapping recommendations (e.g. merging multiple moisture deficit rules) and resolve potential contradictory advice using the precedence hierarchy.

In [ ]:
displayed_recs, all_ranked = prioritize_and_rank_recommendations(
    candidates=candidate_recs,
    intelligence_payload=p3_payload,
    top_n=5,
    tracer=tracer,
)
print(f'Deduplicated from {len(candidate_recs)} candidates down to {len(all_ranked)} unique actions.')
print(f'Top {len(displayed_recs)} selected for primary farmer display.')

Deduplicated from 5 candidates down to 5 unique actions.
Top 5 selected for primary farmer display.


## 9. Transparent Priority Scoring
Display normalized priority scores (0-100) combining trigger severity, evidence strength, actionability, and risk posture.

In [ ]:
score_table = []
for r in all_ranked:
    score_table.append({
        'ID': r['id'],
        'Category': r['category'],
        'Priority Level': r['priority'],
        'Score (0-100)': r['priority_score'],
        'Evidence': r['evidence_strength'],
        'Actionability': r['actionability'],
        'Simulatable': r['what_if_supported'],
    })
score_df = pd.DataFrame(score_table)
print(score_df.to_string(index=False))

             ID     Category Priority Level  Score (0-100) Evidence Actionability  Simulatable
         dq_002 DATA_QUALITY           HIGH             75     HIGH INFORMATIONAL        False
        veg_001   VEGETATION           HIGH             74     HIGH       MONITOR        False
uncertainty_001   MONITORING         MEDIUM             65      LOW INFORMATIONAL        False
       soil_001         SOIL         MEDIUM             53      LOW    ACTIONABLE        False
       risk_002         RISK         MEDIUM             52      LOW INFORMATIONAL        False


## 10. Generate Farmer-Friendly & Technical Views
Compare the clean, non-technical farmer representation with the detailed audit view.

In [ ]:
if displayed_recs:
    farmer_view = format_recommendation_farmer_mode(displayed_recs[0])
    tech_view = format_recommendation_technical_mode(displayed_recs[0])
    print('=== FARMER MODE ===')
    print('Title:', farmer_view['title'])
    print('Why:', farmer_view['reason'])
    print('Action:', farmer_view['action'])
    print('Evidence:', farmer_view['evidence'])
    print('Limitation:', farmer_view['limitations'])
    print('\n=== TECHNICAL MODE AUDIT FIELDS ===')
    print('Source:', tech_view['source'])
    print('Priority Score:', tech_view['priority_score'])
    print('Raw Evidence Signals:', tech_view['raw_evidence'])

=== FARMER MODE ===
Title: Verify flagged data values to improve estimate reliability
Why: Input validation identified missing, imputed, or anomalous input records.
Action: Update or verify the flagged features with on-farm observations to ensure maximum prediction fidelity.
Evidence: ['Identified by rule logic matching field observations.']
Limitation: ['Missing features were imputed using training set medians.']

=== TECHNICAL MODE AUDIT FIELDS ===
Source: None
Priority Score: 75
Raw Evidence Signals: [{'type': 'data_quality_warnings_trigger', 'warning_count': 1, 'warnings': ['Rainfall (1.662354196) is in the extreme 1% tail of historical training data.']}]


## 11. Validate Recommendation Output Contract
Verify all contract fields, non-empty evidence, and absence of prohibited causal terms.

In [ ]:
for rec in displayed_recs:
    fv = format_recommendation_farmer_mode(rec)
    valid, rec_issues = validate_single_recommendation(fv)
    assert valid, f"Validation failed: {rec_issues}"
print('All displayed recommendations passed schema and safety validation.')

All displayed recommendations passed schema and safety validation.


## 12. Demonstration Across Key Agricultural Scenarios
Evaluate the recommendation engine on diverse farm states: moisture stress, heat wave, high uncertainty, and OOD.

In [ ]:
scenarios = {
    'Normal Conditions': {'soil_moisture': 28.0, 'NDVI': 0.70, 'rainfall': 15.0, 'temperature': 24.0},
    'Severe Moisture Deficit': {'soil_moisture': 10.0, 'rainfall': 1.0, 'temperature': 34.0},
    'Vegetation Canopy Decline': {'NDVI': 0.22, 'GNDVI': 0.20, 'SAVI': 0.18},
    'Out-of-Distribution Weather': {'temperature': 70.0, 'rainfall': 500.0},
}

scenario_results = []
for sname, sinputs in scenarios.items():
    row = sample_input.copy()
    row.update(sinputs)
    p3_res = analyze_crop_prediction(row)
    p4_res = generate_recommendations(intelligence_payload=p3_res, input_data=row)
    top_title = p4_res['recommendations'][0]['title'] if p4_res['recommendations'] else 'None'
    top_cat = p4_res['recommendations'][0]['category'] if p4_res['recommendations'] else 'N/A'
    scenario_results.append({
        'Scenario': sname,
        'Risk Level': p3_res['risk']['level'],
        'Generated': p4_res['summary']['total_generated'],
        'Top Action Category': top_cat,
        'Top Action Title': top_title,
    })

res_df = pd.DataFrame(scenario_results)
print(res_df.to_string(index=False))

                   Scenario Risk Level  Generated Top Action Category                                                      Top Action Title
          Normal Conditions        LOW          1          VEGETATION                   Healthy canopy greenness supporting yield potential
    Severe Moisture Deficit   MODERATE          7               WATER                             Review soil moisture and irrigation needs
  Vegetation Canopy Decline   MODERATE          5        DATA_QUALITY            Verify flagged data values to improve estimate reliability
Out-of-Distribution Weather   MODERATE          7        DATA_QUALITY Data quality alert: Inputs fall outside typical model training bounds


## 13. Final Recommendation JSON Output Contract
Export complete API-ready JSON structure for Phase 5 (What-If Simulator) and Phase 6 (FastAPI).

In [ ]:
final_output = generate_recommendations(
    intelligence_payload=p3_payload,
    input_data=sample_input,
    top_n=3,
)
print(json.dumps(final_output['summary'], indent=2))
print('\nTop Recommendation JSON:')
print(json.dumps(final_output['recommendations'][0], indent=2))

{
  "total_generated": 5,
  "displayed": 3,
  "high_priority": 2,
  "medium_priority": 1,
  "low_priority": 0,
  "crop_specific_recommendations_available": false,
  "executive_summary": "Estimated Rice yield is 36.43 unconfirmed with an assessed MODERATE yield risk. The primary recommended action is to update or verify the flagged features with on-farm observations to ensure maximum prediction fidelity. Because model uncertainty is high across ensemble trees, verify field ground conditions before committing to major inputs."
}

Top Recommendation JSON:
{
  "id": "dq_002",
  "title": "Verify flagged data values to improve estimate reliability",
  "category": "DATA_QUALITY",
  "priority": "HIGH",
  "actionability": "INFORMATIONAL",
  "summary": "Input validation identified missing, imputed, or anomalous input records.",
  "reason": "Input validation identified missing, imputed, or anomalous input records.",
  "action": "Update or verify the flagged features with on-farm observations to e